In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/src/task4_enriched_profit

In [0]:
import pytest
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from datetime import date
from decimal import Decimal

In [0]:
# Setup: mimic profit_by_year table creation (without calling main_enrich)
enriched_orders = spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
profit_df = build_profit_by_year(enriched_orders)
profit_df = dedupe(profit_df)
delta_writer(profit_df, "az_ci_de_dbs.ecom_dev.profit_by_year")


# --- Test: Source table exists ---

def test_enriched_orders_exists():
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.enriched_orders"), "enriched_orders table does not exist"


def test_enriched_orders_not_empty():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.enriched_orders")
    assert df.count() > 0, "enriched_orders table is empty"


# --- Test: build_profit_by_year output ---

def test_build_profit_by_year_not_empty():
    result = build_profit_by_year(enriched_orders)
    assert result.count() > 0, "build_profit_by_year returned empty DataFrame"


def test_build_profit_by_year_expected_columns():
    result = build_profit_by_year(enriched_orders)
    expected_cols = ["Year", "product_category", "product_sub_category", "customer", "total_profit"]
    assert sorted(result.columns) == sorted(expected_cols), f"Expected {sorted(expected_cols)}, got {sorted(result.columns)}"


def test_build_profit_by_year_no_extra_columns():
    result = build_profit_by_year(enriched_orders)
    assert len(result.columns) == 5, f"Expected 5 columns, got {len(result.columns)}"


def test_build_profit_by_year_year_is_integer():
    result = build_profit_by_year(enriched_orders)
    year_type = str(result.schema["Year"].dataType)
    assert "int" in year_type.lower(), f"Year column should be integer type, got {year_type}"


def test_build_profit_by_year_year_valid_range():
    result = build_profit_by_year(enriched_orders)
    min_year = result.agg(F.min("Year")).collect()[0][0]
    max_year = result.agg(F.max("Year")).collect()[0][0]
    assert min_year >= 2000, f"Year too low: {min_year}"
    assert max_year <= 2030, f"Year too high: {max_year}"


def test_build_profit_by_year_no_null_groupby_cols():
    result = build_profit_by_year(enriched_orders)
    for col_name in ["Year", "product_category", "product_sub_category", "customer"]:
        nulls = result.filter(F.col(col_name).isNull()).count()
        assert nulls == 0, f"Found {nulls} nulls in {col_name}"


def test_build_profit_by_year_no_null_total_profit():
    result = build_profit_by_year(enriched_orders)
    nulls = result.filter(F.col("total_profit").isNull()).count()
    assert nulls == 0, f"Found {nulls} nulls in total_profit"


def test_build_profit_by_year_profit_rounded_to_2_places():
    """total_profit should be rounded to 2 decimal places."""
    result = build_profit_by_year(enriched_orders)
    violations = result.filter(
        F.col("total_profit") != F.round(F.col("total_profit"), 2)
    ).count()
    assert violations == 0, f"Found {violations} rows with profit not rounded to 2 places"


def test_build_profit_by_year_allows_negative_profit():
    """Profit can be negative (losses) - should be retained."""
    result = build_profit_by_year(enriched_orders)
    neg_count = result.filter(F.col("total_profit") < 0).count()
    assert neg_count >= 0, "Negative profit check failed"


def test_build_profit_by_year_aggregation_reduces_rows():
    """Aggregation should reduce or keep row count vs source."""
    result = build_profit_by_year(enriched_orders)
    assert result.count() <= enriched_orders.count(), "Aggregation should not increase row count"


def test_build_profit_by_year_no_duplicates():
    """Each (Year, product_category, product_sub_category, customer) combination should be unique."""
    result = build_profit_by_year(enriched_orders)
    total = result.count()
    distinct = result.select("Year", "product_category", "product_sub_category", "customer").distinct().count()
    assert total == distinct, f"Duplicate groups found: {total} rows, {distinct} distinct groups"


# --- Test: profit_by_year table ---

def test_profit_by_year_table_exists():
    assert spark.catalog.tableExists("az_ci_de_dbs.ecom_dev.profit_by_year"), "profit_by_year table does not exist"


def test_profit_by_year_table_not_empty():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.profit_by_year")
    assert df.count() > 0, "profit_by_year table is empty"


def test_profit_by_year_table_no_duplicates():
    df = spark.read.table("az_ci_de_dbs.ecom_dev.profit_by_year")
    total = df.count()
    distinct = df.dropDuplicates().count()
    assert total == distinct, f"Duplicates in profit_by_year: {total} rows, {distinct} distinct"


def test_profit_by_year_table_matches_build_output():
    """Table row count should match build output after dedupe."""
    table_count = spark.read.table("az_ci_de_dbs.ecom_dev.profit_by_year").count()
    build_count = dedupe(build_profit_by_year(enriched_orders)).count()
    assert table_count == build_count, f"Table count ({table_count}) != build output ({build_count})"


# --- Run all tests ---

def main_enriched_profit_test():
    test_functions = [
        # source table
        test_enriched_orders_exists,
        test_enriched_orders_not_empty,
        # build_profit_by_year
        test_build_profit_by_year_not_empty,
        test_build_profit_by_year_expected_columns,
        test_build_profit_by_year_no_extra_columns,
        test_build_profit_by_year_year_is_integer,
        test_build_profit_by_year_year_valid_range,
        test_build_profit_by_year_no_null_groupby_cols,
        test_build_profit_by_year_no_null_total_profit,
        test_build_profit_by_year_profit_rounded_to_2_places,
        test_build_profit_by_year_allows_negative_profit,
        test_build_profit_by_year_aggregation_reduces_rows,
        test_build_profit_by_year_no_duplicates,
        # profit_by_year table
        test_profit_by_year_table_exists,
        test_profit_by_year_table_not_empty,
        test_profit_by_year_table_no_duplicates,
        test_profit_by_year_table_matches_build_output,
    ]

    passed, failed = 0, 0
    for fn in test_functions:
        try:
            fn()
            passed += 1
            print(f"  [PASSED] {fn.__name__}")
        except Exception as e:
            failed += 1
            print(f"  [FAILED] {fn.__name__}: {e}")

    print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")

    # Cleanup: drop mimicked table
    spark.sql("DROP TABLE IF EXISTS az_ci_de_dbs.ecom_dev.profit_by_year")
    print("\n[CLEANUP] Dropped az_ci_de_dbs.ecom_dev.profit_by_year")